In [2]:
import numpy as np

def sigmoid(x):
    return 1/(1+np.exp(-x))

def dsigmoid(x):
    return sigmoid(x) * (1- sigmoid(x)) #derivative oof sigmoid this help us compute the change in output how much the output of the sigmoid function changes with respect to its input

def tanh(x):
    return np.tanh(x)

def dtanh(x):
    return 1 - tanh(x) ** 2

In [3]:
def initialize_parameters(input_size, hidden_size):
    params = {}

    params['wf'] = np.random.randn(hidden_size, hidden_size + input_size) * 0.01
    params['bf'] = np.zeros((hidden_size, 1))

    params['wi'] = np.random.randn(hidden_size, hidden_size + input_size) * 0.01
    params['bi'] = np.zeros((hidden_size, 1))

    params['wc'] = np.random.randn(hidden_size, hidden_size + input_size) * 0.01
    params['bc'] = np.zeros((hidden_size, 1))

    params['wo'] = np.random.randn(hidden_size, hidden_size + input_size) * 0.01
    params['bo'] = np.zeros((hidden_size, 1))

    return params


In [4]:
def lstm_cell_forward(x_t,c_prev, h_prev, params): #current input, cell state previous, hidden state previous

    #parameter extraction
    wf, bf=params['wf'], params['bf']#forget
    wi, bi=params['wi'], params['bi']#input
    wc, bc=params['wc'], params['bc']#cell
    wo, bo=params['wo'], params['bo']#output

    #combine h_prev has previous context and x_t is new word and together it will make a meaning
    combined= np.vstack((h_prev, x_t))

    #forget gate
    f_t=sigmoid(np.dot(wf, combined)+bf)

    #input gate 
    i_t=sigmoid(np.dot(wi, combined)+bi)

    #candidate gate
    ĉ_t = tanh(np.dot(wc, combined) + bc)

    #update candidate
    C_t= f_t*c_prev + i_t*ĉ_t  #core of the lstm this decide what to remember an what to forget

    #output gate
    o_t=sigmoid(np.dot(wo, combined)+bo)

    #update hidden gate
    h_t= o_t * tanh(C_t) #tanh because C_t may have neg(-) vals and this tanh  squashes it to [-1, 1], keeping that signed information.

    cache= (f_t, i_t, ĉ_t, o_t, C_t, c_prev, h_prev, x_t, combined, wf, wi, wc, wo)

    return h_t, C_t, cache

In [5]:
def lstm_cell_backward(dh_t, dc_t, cache):
    #unpacking
    f_t, i_t, ĉ_t, o_t, C_t, c_prev, h_prev, x_t, combined, wf, wi, wc, wo = cache

    #C_t may vary - to + so squash
    tanh_C_t =np.tanh(C_t)

     # Step 1: Compute output gate gradient
    do_t = dh_next * tanh_C_t
    do_raw = do_t * o_t * (1 - o_t)  # dsigmoid

    # Step 2: Compute cell state gradient
    dC_t = dh_next * o_t * (1 - tanh_C_t**2) + dc_next

    # Step 3: Compute candidate cell gradient
    dĉ_t = dC_t * i_t
    dĉ_raw = dĉ_t * (1 - ĉ_t ** 2)  # dtanh

    # Step 4: Compute input gate gradient
    di_t = dC_t * ĉ_t
    di_raw = di_t * i_t * (1 - i_t)  # dsigmoid

    # Step 5: Compute forget gate gradient
    df_t = dC_t * c_prev
    df_raw = df_t * f_t * (1 - f_t)  # dsigmoid

    # Step 6: Compute parameter gradients
    dWf = np.dot(df_raw, combined.T)
    dWi = np.dot(di_raw, combined.T)
    dWc = np.dot(dĉ_raw, combined.T)
    dWo = np.dot(do_raw, combined.T)

    dbf = df_raw
    dbi = di_raw
    dbc = dĉ_raw
    dbo = do_raw

    # Step 7: Compute gradients w.r.t input and previous hidden state
    d_combined = (
        np.dot(wf.T, df_raw)
        + np.dot(wi.T, di_raw)
        + np.dot(wc.T, dĉ_raw)
        + np.dot(wo.T, do_raw)
    )

    dh_prev = d_combined[:h_prev.shape[0], :]
    dx_t = d_combined[h_prev.shape[0]:, :]
    dc_prev = dC_t * f_t

    # Return all gradients
    grads = {
        "dWf": dWf, "dbf": dbf,
        "dWi": dWi, "dbi": dbi,
        "dWc": dWc, "dbc": dbc,
        "dWo": dWo, "dbo": dbo,
        "dx": dx_t,
        "dh_prev": dh_prev,
        "dc_prev": dc_prev
    }

    return grads
